In [ ]:
######  LTV测算：API用户 更新说明
### 需注意两部分修改：
### 1.搜索修改后缀_lss 表名为你自己的表名, 如ltv_first_loan_driver_lss 修改为 ltv_first_loan_driver_你的名字
### 2.修改文件夹路径为你自己的路径：
###   

In [2]:
# 导入库，已内置在query_analysis_tool中
import query_analysis_tool as qat
import pandas as pd
import numpy as np
import importlib
importlib.reload(qat)  # 强制重新加载最新的包代码

## 修改后缀_lss 表名为你自己的表名, 如cash_flow_driver_lss 修改为 cash_flow_driver_你的名字
ltv_driver_table = 'ltv_api_driver_lss'
ltv_result_table = 'ltv_api_result_lss'

## 自定义bizdate 与数据统计区间
from datetime import date, timedelta
bizdate = (date.today() - timedelta(days=1)).strftime("%Y%m%d")  #设置为昨天
begin_date = '2024-01-01'  
# end_date = '2025-12-18'

#文件路径
file_path = r"D:\10.LTV月度更新\API部分\LTV_API渠道模版制作.xlsx"

# 人群与sheet映射（按你的实际sheet名改）
population_to_sheet = {
    "API整体": "API整体",

}

#### LTV主数据准备

In [ ]:
#=========================
#选出要看的首贷订单客群，必须的基础要素（app_user_id, cust_no, order_number, register_time, loan_time, population_category）
#=========================
drop_driver = '''
DROP TABLE IF EXISTS {ltv_driver_table};
'''.format(ltv_driver_table=ltv_driver_table)

driver_query ='''
CREATE TABLE {ltv_driver_table} AS

SELECT  app_user_id
       ,cust_no
       ,order_number
       ,loan_time
	   ,m2.mobile
	   ,m2.id_card_number
       ,'API整体' AS population_category
FROM
(
	SELECT  user_no    AS app_user_id
	       ,cust_no
	       ,loan_time
	       ,order_number
	       ,loan_amt
	       ,period
	       ,utm_source AS order_utm_source
	       ,fee_rate
	       ,inner_app
	FROM xyf_dws.dws_inloan_user_order_df
	WHERE pt = MAX_PT('xyf_dws.dws_inloan_user_order_df')
	AND app IN ('xyf01')
	AND business_line = 'API' --API首贷 
	AND loan_flag = '首贷'
	AND loan_status = 'success'
	AND DATE(loan_time) >= '2024-01-01'
	AND inner_app NOT IN ('fxk_hexj', 'fxk_aj360', 'fxk_zyxj', 'fxk01_360zybx', 'fxk01_360zyaj') 
) first_loan
LEFT JOIN
(
	SELECT  app_user_id
	       ,mobile
	       ,id_card_number
	FROM xyf_dim.dim_user_app_basic_info_df
	WHERE pt = MAX_PT('xyf_dim.dim_user_app_basic_info_df') --and current_utm_source LIKE '%DY%' 
	AND app IN ('xyf01', 'fxk')
	GROUP BY  app_user_id
	         ,mobile
	         ,id_card_number
) m2
ON first_loan.user_no = m2.app_user_id
LEFT JOIN
(
	SELECT  abbr
	       ,name
	FROM xyf_ods.ods_sfy_sta_channel_df
	WHERE pt = MAX_PT('xyf_ods.ods_sfy_sta_channel_df') 
) b
ON first_loan.order_utm_source = b.abbr

'''.format(ltv_driver_table=ltv_driver_table)

##创建driver,创建app_user X mob level的现金流数据。
qat.execute_sql(drop_driver)
qat.execute_sql(driver_query)


⚙️ 正在执行 SQL (无结果返回): DROP TABLE IF EXISTS ltv_api_driver_lss; ...
✅ SQL 执行完成！
⚙️ 正在执行 SQL (无结果返回): CREATE TABLE ltv_api_driver_lss AS  SELECT  app_user_id      ...
✅ SQL 执行完成！


In [ ]:
SELECT  app_user_id
       ,cust_no
       ,order_number
       ,loan_time
       ,'API整体' AS population_category
FROM
(
	SELECT  user_no                                                   AS app_user_id
           ,cust_no
	       ,loan_time                                                 
	       ,order_number
	       ,loan_amt                                                  
	       ,period
	       ,utm_source                                                AS order_utm_source
	       ,fee_rate
	       ,inner_app
	FROM xyf_dws.dws_inloan_user_order_df
	WHERE pt = MAX_PT('xyf_dws.dws_inloan_user_order_df') 
	AND app IN ('xyf01') 
    AND business_line = 'API'  --API首贷
    AND loan_flag = '首贷' 
	AND loan_status = 'success'
	AND inner_app NOT IN ('fxk_hexj', 'fxk_aj360', 'fxk_zyxj', 'fxk01_360zybx', 'fxk01_360zyaj') 
) first_loan
LEFT JOIN
(
	SELECT  abbr
	       ,name
	FROM xyf_ods.ods_sfy_sta_channel_df
	WHERE pt = MAX_PT('xyf_ods.ods_sfy_sta_channel_df') 
) b
ON first_loan.order_utm_source = b.abbr

UNION ALL

SELECT  app_user_id
       ,cust_no
       ,order_number
       ,loan_time
       ,'API整体' AS population_category
FROM
(
	SELECT  user_no                                                   AS app_user_id
           ,cust_no
	       ,loan_time                                                 
	       ,order_number
	       ,loan_amt                                                  
	       ,period
	       ,utm_source                                                AS order_utm_source
	       ,fee_rate
	       ,inner_app
	FROM xyf_dws.dws_inloan_user_order_df
	WHERE pt = MAX_PT('xyf_dws.dws_inloan_user_order_df') 
	AND app IN ('xyf01') 
    AND business_line = 'API'  --API首贷
    AND loan_flag = '首贷' 
	AND loan_status = 'success'
	AND inner_app NOT IN ('fxk_hexj', 'fxk_aj360', 'fxk_zyxj', 'fxk01_360zybx', 'fxk01_360zyaj') 
) first_loan
LEFT JOIN
(
	SELECT  abbr
	       ,name
	FROM xyf_ods.ods_sfy_sta_channel_df
	WHERE pt = '${bizdate}' 
) b
ON first_loan.order_utm_source = b.abbr

-- 随后加上别的渠道,感觉也不用,后续好多36资方没有了



#### 首贷期限、定价、件均(复贷统一使用大盘的复贷情况，复贷代码已注释，特定客群使用真实复贷情况的话可复用)

In [ ]:
query = '''
--=========================================================================================
--无特殊业务含义。做一个view，这样可以只改这里的driver 
--=========================================================================================
WITH
acquired_customer AS
(
SELECT  *
FROM {ltv_driver_table}  --driver  here<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
) , 

first_loan AS
(
SELECT  
        first_loan.loan_time                  AS first_loan_time             
       ,SUBSTR(first_loan.loan_time, 1, 7)    AS first_loan_month
       ,first_loan.user_no
       ,first_loan.cust_no
       ,first_loan.first_order_number
       ,first_loan.loan_amt                   AS first_loan_amt
       ,first_loan.period
	   ,first_loan.fee_rate
       ,acquired_customer.population_category
FROM acquired_customer
INNER JOIN
(
	SELECT  user_no
	       ,cust_no
	       ,first_order_number
	       ,loan_amt
	       ,period
	       ,fee_rate
	       ,first_order_time
	       ,loan_time
	       ,loan_flag
	FROM xyf_dws.dws_inloan_user_order_df
	WHERE pt = MAX_PT('xyf_dws.dws_inloan_user_order_df')
	AND app IN ('xyf01')
	AND business_line = 'API'  --API首贷 
	AND loan_status = 'success'
	AND loan_flag = '首贷' 
) first_loan
ON acquired_customer.app_user_id = first_loan.user_no
)

SELECT  shoudai.first_loan_month          AS 首贷放款月
       ,shoudai.*EXCEPT(first_loan_month) 
       --,fudai.*EXCEPT(first_loan_month, population_category)
FROM
(
	SELECT  first_loan_month
	       ,population_category
	       ,COUNT(DISTINCT cust_no)                                AS 放款人数
	       ,SUM(first_loan_amt)/10000                              AS `首贷放款(万)`
	       ,SUM(first_loan_amt)/COUNT(DISTINCT first_order_number) AS 首贷件均
	       ,SUM(first_loan_amt * fee_rate)/SUM(first_loan_amt)     AS 首贷定价
	       ,SUM(first_loan_amt * period)/SUM(first_loan_amt)       AS 首贷期限
	FROM first_loan
	GROUP BY  first_loan_month
	         ,population_category
) shoudai
LEFT JOIN
(
	SELECT  first_loan.first_loan_month
	       ,first_loan.population_category
	       ,COUNT(DISTINCT fudai_orders.cust_no)                                          AS 复贷放款人数
	       ,SUM(fudai_orders.loan_amt)/10000                                              AS `复贷放款(万)`
	       ,SUM(fudai_orders.loan_amt)/COUNT(DISTINCT fudai_orders.first_order_number)    AS 复贷件均
	       ,SUM(fudai_orders.loan_amt * fudai_orders.fee_rate)/SUM(fudai_orders.loan_amt) AS 复贷定价
	       ,SUM(fudai_orders.loan_amt * fudai_orders.period)/SUM(fudai_orders.loan_amt)   AS 复贷期限
	FROM first_loan
	INNER JOIN
	(
		SELECT  order_number
		       ,user_no
		       ,cust_no
		       ,first_order_number
		       ,app
		       ,inner_app
		       ,business_line
		       ,loan_flag
		       ,first_order_time
		       ,loan_time
		       ,loan_amt
		       ,period
		       ,asset_type_flag
		       ,fee_rate
		       ,biz_flow_number
		FROM xyf_dws.dws_inloan_user_order_df
		WHERE pt = MAX_PT('xyf_dws.dws_inloan_user_order_df')
		AND loan_status = 'success'
		AND app IN ('xyf01', 'fxk')
		AND loan_flag <> '首贷'
		AND business_line IN ('APP', '小程序端') -- APP上的复贷订单 
 
	) fudai_orders
	ON first_loan.cust_no = fudai_orders.cust_no AND first_loan.first_loan_time <= fudai_orders.loan_time
	GROUP BY  first_loan.first_loan_month
	         ,first_loan.population_category
) fudai
ON shoudai.first_loan_month = fudai.first_loan_month AND shoudai.population_category = fudai.population_category
'''.format(ltv_driver_table=ltv_driver_table)

data_stats = qat.run_query(query)

正在获取数据，首段 SQL: 
--=============================================== ...


In [5]:
# 循环每个人群写入对应sheet
for pop, sheet in population_to_sheet.items():
    sub = data_stats.loc[data_stats["population_category"] == pop].copy()
    if sub.empty:
        print(f"[跳过] {pop}: 无数据")
        continue
    
    sub["首贷放款月"] = sub["首贷放款月"].astype(str)
    sub = sub.sort_values(by='首贷放款月')
    df_t_pop = sub.T.reset_index()
    
    qat.write_dataframe_to_template_area(
        file_path=file_path,
        sheet_name=sheet,
        df=df_t_pop,
        start_cell="B3",          # 期限、定价初始位置
        include_header=False,
        clear_before_write=False,  
    )

    print(f"[完成] {pop} -> {sheet}")


[OK] 写入 12 行 x 31 列 -> sheet=API整体 B3
[完成] API整体 -> API整体


#### 复贷大盘期限、定价、件均（API均使用统一的复贷）

In [3]:
query = '''
--=====================================================================================
--api首贷的人在app的复贷
--=====================================================================================
WITH first_loan AS
(
	SELECT  m2.mobile
	       ,app
	       ,firstloan_time
	       ,utm_source
	FROM
	(
		SELECT  user_no
		       ,app
		       ,loan_time AS firstloan_time
		       ,utm_source
		FROM xyf_dws.dws_inloan_user_order_df
		WHERE pt = MAX_PT('xyf_dws.dws_inloan_user_order_df')
		AND business_line = 'API'
		AND app IN ('xyf01')
		AND loan_flag = '首贷'
		AND loan_status = 'success'
		AND inner_app NOT IN ('fxk_hexj', 'fxk_aj360', 'fxk_zyxj', 'fxk01_360zybx', 'fxk01_360zyaj') 
	) m1
	LEFT JOIN
	(
		SELECT  app_user_id
		       ,mobile
		FROM xyf_dim.dim_user_app_basic_info_df
		WHERE pt = MAX_PT('xyf_dim.dim_user_app_basic_info_df') --and current_utm_source LIKE '%DY%' 
		AND app IN ('xyf01', 'fxk')
		GROUP BY  app_user_id
		         ,mobile
	) m2
	ON m1.user_no = m2.app_user_id
) --api ever首贷用户 

SELECT  fudai_orders.loan_month
       ,COUNT(DISTINCT fudai_orders.cust_no)                                          AS 复贷放款人数
       ,SUM(fudai_orders.loan_amt)/10000                                              AS `复贷放款(万)`
       ,SUM(fudai_orders.loan_amt)/COUNT(DISTINCT fudai_orders.order_number)          AS 复贷件均
       ,SUM(fudai_orders.loan_amt * fudai_orders.fee_rate)/SUM(fudai_orders.loan_amt) AS 复贷定价
       ,SUM(fudai_orders.loan_amt * fudai_orders.period)/SUM(fudai_orders.loan_amt)   AS 复贷期限
FROM
(
	SELECT  first_loan.utm_source
	       ,fudai.order_number
	       ,fudai.cust_no
	       ,fudai.mobile
	       ,first_loan.firstloan_time
	       ,fudai.loan_time
	       ,fudai.loan_amt
	       ,fudai.period
	       ,fudai.loan_month
	       ,fudai.fee_rate
	FROM first_loan
	INNER JOIN
	(
		SELECT  m2.mobile
		       ,m1.cust_no
		       ,m1.app
		       ,m1.loan_time
		       ,m1.order_number
		       ,m1.loan_month
		       ,m1.loan_amt
		       ,m1.period
		       ,m1.fee_rate
		FROM
		(
			SELECT  user_no
			       ,cust_no
			       ,app
			       ,loan_time
			       ,order_number
			       ,SUBSTR(loan_time,1,7) AS loan_month
			       ,loan_amt
			       ,period
			       ,fee_rate
			FROM xyf_dws.dws_inloan_user_order_df
			WHERE pt = MAX_PT('xyf_dws.dws_inloan_user_order_df')
			AND business_line IN ('APP', '小程序端')
			AND app IN ('xyf01')
			AND loan_flag <> '首贷'
			AND loan_status = 'success'
			AND DATE(loan_time) >= '2024-01-01' 
			AND inner_app NOT IN ('fxk_hexj', 'fxk_aj360', 'fxk_zyxj', 'fxk01_360zybx', 'fxk01_360zyaj') 
		) m1
		LEFT JOIN
		(
			SELECT  app_user_id
			       ,mobile
			       ,id_card_number
			FROM xyf_dim.dim_user_app_basic_info_df
			WHERE pt = MAX_PT('xyf_dim.dim_user_app_basic_info_df') --and current_utm_source LIKE '%DY%' 
			AND app IN ('xyf01', 'fxk')
			GROUP BY  app_user_id
			         ,mobile
			         ,id_card_number
		) m2
		ON m1.user_no = m2.app_user_id
	) fudai
	ON first_loan.mobile = fudai.mobile AND first_loan.app = fudai.app
) fudai_orders
GROUP BY  fudai_orders.loan_month
ORDER BY  fudai_orders.loan_month
'''
data_stats = qat.run_query(query)
data_stats["loan_month"] = data_stats["loan_month"].astype(str)
data_stats = data_stats.sort_values(by='loan_month')

正在获取数据，首段 SQL: 
--=============================================== ...


In [4]:
# 循环每个人群写入对应sheet，排除尊享贷
for pop, sheet in population_to_sheet.items():
    if pop == "尊享贷": 
        continue

    sub = data_stats.drop(columns=["loan_month"]).copy()
    if sub.empty:
        print(f"[跳过] {pop}: 无数据")
        continue

    df_t_pop = sub.T.reset_index()
    
    qat.write_dataframe_to_template_area(
        file_path=file_path,
        sheet_name=sheet,
        df=df_t_pop,
        start_cell="B10",          
        include_header=False,
        clear_before_write=False,  
    )

    print(f"[完成] {pop} -> {sheet}")

[OK] 写入 5 行 x 31 列 -> sheet=API整体 B10
[完成] API整体 -> API整体


#### API2APP成交人数&成交人均（API授信，APP首贷）

In [5]:
query = '''
SELECT  loan_month
       --,CASE WHEN inner_app IN ('xyf01_br','xyf01_br02') THEN '百融'  ELSE name END AS name
       --,SUM(loan_amount)                                                          AS loan_amt
       ,COUNT(DISTINCT order_number)                                              AS app成交人数
	   ,SUM(loan_amount)/COUNT(DISTINCT order_number)                             AS app成交人均
FROM
(
	SELECT  utm_source
	       ,order_number
	       ,t1.mobile
	       ,loan_amount
	       ,loan_month
	       ,inner_app
	FROM
	(
		SELECT  *
		FROM
		(
			SELECT  mobile
			       ,utm_source
			       ,ROW_NUMBER() OVER (PARTITION BY mobile ORDER BY  row_crt_ts ASC ) AS rn
			       ,inner_app
			FROM xyf_ods.ods_xyf_bi_cash_activation_log_df
			WHERE pt = max_pt('xyf_ods.ods_xyf_bi_cash_activation_log_df')
			AND activation_status = 'success'
			AND activation_source <> 'fxk_copy'
			AND inner_app <> app 
		)
		WHERE rn = 1 
	) t1
	LEFT JOIN
	(
		SELECT  m2.mobile
		       ,user_no
		       ,order_number
		       ,loan_amount
		       ,draw_apply_time
		       ,loan_date
		       ,loan_month
		       ,id_card_number
		FROM
		(
			SELECT  user_no
			       ,order_number
			       ,loan_amt              AS loan_amount
			       ,first_order_time      AS draw_apply_time
			       ,TO_DATE(loan_time)    AS loan_date
			       ,SUBSTR(loan_time,1,7) AS loan_month
			FROM xyf_dws.dws_inloan_user_order_df
			WHERE pt = MAX_PT('xyf_dws.dws_inloan_user_order_df')
			AND app = lower(inner_app)
			AND app IN ('xyf01')
			AND loan_flag = '首贷'
			AND loan_status = 'success' 
		) m1
		LEFT JOIN
		(
			SELECT  app_user_id
			       ,mobile
			       ,id_card_number
			FROM xyf_dim.dim_user_app_basic_info_df
			WHERE pt = MAX_PT('xyf_dim.dim_user_app_basic_info_df')
			AND app IN ('xyf01', 'fxk')
			GROUP BY  app_user_id
			         ,mobile
			         ,id_card_number
		) m2
		ON m1.user_no = m2.app_user_id
	) t2
	ON t1.mobile = t2.mobile
) a
LEFT JOIN
(
	SELECT  abbr
	       ,name
	FROM xyf_ods.ods_sfy_sta_channel_df
	WHERE pt = max_pt('xyf_ods.ods_sfy_sta_channel_df') 
) b
ON a.utm_source = b.abbr
WHERE loan_month >= '2024-01'
GROUP BY  loan_month
         --,CASE WHEN inner_app IN ('xyf01_br','xyf01_br02') THEN '百融'  ELSE name END


'''.format(ltv_driver_table=ltv_driver_table)

data_stats = qat.run_query(query)

data_stats["loan_month"] = data_stats["loan_month"].astype(str)
data_stats = data_stats.sort_values(by='loan_month')

正在获取数据，首段 SQL: 
SELECT  loan_month
       --,CASE WHEN inner_app  ...


In [9]:
sub = data_stats.drop(columns=["loan_month"]).copy()
df_t_pop = sub.T.reset_index()
qat.write_dataframe_to_template_area(
    file_path=file_path,
    sheet_name=sheet,
    df=df_t_pop,
    start_cell="B18",          # 按你的模板位置调整
    include_header=False,
    clear_before_write=False,  
)

# # 循环每个人群写入对应sheet
# for pop, sheet in population_to_sheet.items():
#     sub = data_stats.loc[data_stats["population_category"] == pop,["app成交人数","app成交人均"]].copy()
#     if sub.empty:
#         print(f"[跳过] {pop}: 无数据")
#         continue

#     df_t_pop = sub.T.reset_index()
    
#     qat.write_dataframe_to_template_area(
#         file_path=file_path,
#         sheet_name=sheet,
#         df=df_t_pop,
#         start_cell="B18",          # 按你的模板位置调整
#         include_header=False,
#         clear_before_write=False,  
#     )

#     print(f"[完成] {pop} -> {sheet}")

[OK] 写入 2 行 x 31 列 -> sheet=API整体 B18


[{'sheet_name': 'API整体',
  'start_cell': 'B18',
  'rows': 2,
  'cols': 31,
  'skipped': False}]

#### 复贷系数

#### 暂时先用excle模版

In [2]:
query = '''
--=========================================================================================================================================================
--LTV 复贷系数。 假设订单表宽表首复贷标签正确。在看复贷系数时，首贷以后，用cust_no去看所有复贷。
--首贷归因。首贷归因基于首贷user_no归因。一个cust_no 可以注册多个手机号，有多个user_no。但这里user_no贷款被认定为首贷（前面假设），认为这个注册及首贷是新用户。
--这种做法潜在意思就是 以订单宽表首复贷标签为准来判断是否为新客/老客。
--==========================================================================================================================================================

WITH
--===========================================================================
--首贷,后续的复贷loan_amt sum 30,60,90...   user_no unique
--===========================================================================
cust_level_revolving AS(
SELECT  first_loan.user_no
       ,first_loan.cust_no
       ,substr(first_loan.loan_time,1,7) AS first_loan_month
       ,DATE(first_loan.loan_time)       AS first_loan_date
       ,first_loan.loan_amt              AS first_loan_amt
       --复贷金额 
       ,SUM(CASE WHEN DATEDIFF(DATE(loans.loan_time),DATE(first_loan.loan_time)) BETWEEN 0 AND 30 * 1 THEN loans.loan_amt END) AS reloan_amt1
       ,SUM(CASE WHEN DATEDIFF(DATE(loans.loan_time),DATE(first_loan.loan_time)) BETWEEN 0 AND 30 * 2 THEN loans.loan_amt END) AS reloan_amt2
       ,SUM(CASE WHEN DATEDIFF(DATE(loans.loan_time),DATE(first_loan.loan_time)) BETWEEN 0 AND 30 * 3 THEN loans.loan_amt END) AS reloan_amt3
       ,SUM(CASE WHEN DATEDIFF(DATE(loans.loan_time),DATE(first_loan.loan_time)) BETWEEN 0 AND 30 * 4 THEN loans.loan_amt END) AS reloan_amt4
       ,SUM(CASE WHEN DATEDIFF(DATE(loans.loan_time),DATE(first_loan.loan_time)) BETWEEN 0 AND 30 * 5 THEN loans.loan_amt END) AS reloan_amt5
       ,SUM(CASE WHEN DATEDIFF(DATE(loans.loan_time),DATE(first_loan.loan_time)) BETWEEN 0 AND 30 * 6 THEN loans.loan_amt END) AS reloan_amt6
       ,SUM(CASE WHEN DATEDIFF(DATE(loans.loan_time),DATE(first_loan.loan_time)) BETWEEN 0 AND 30 * 7 THEN loans.loan_amt END) AS reloan_amt7
       ,SUM(CASE WHEN DATEDIFF(DATE(loans.loan_time),DATE(first_loan.loan_time)) BETWEEN 0 AND 30 * 8 THEN loans.loan_amt END) AS reloan_amt8
       ,SUM(CASE WHEN DATEDIFF(DATE(loans.loan_time),DATE(first_loan.loan_time)) BETWEEN 0 AND 30 * 9 THEN loans.loan_amt END) AS reloan_amt9
       ,SUM(CASE WHEN DATEDIFF(DATE(loans.loan_time),DATE(first_loan.loan_time)) BETWEEN 0 AND 30 * 10 THEN loans.loan_amt END) AS reloan_amt10
       ,SUM(CASE WHEN DATEDIFF(DATE(loans.loan_time),DATE(first_loan.loan_time)) BETWEEN 0 AND 30 * 11 THEN loans.loan_amt END) AS reloan_amt11
       ,SUM(CASE WHEN DATEDIFF(DATE(loans.loan_time),DATE(first_loan.loan_time)) BETWEEN 0 AND 30 * 12 THEN loans.loan_amt END) AS reloan_amt12
       ,SUM(CASE WHEN DATEDIFF(DATE(loans.loan_time),DATE(first_loan.loan_time)) BETWEEN 0 AND 30 * 13 THEN loans.loan_amt END) AS reloan_amt13
       ,SUM(CASE WHEN DATEDIFF(DATE(loans.loan_time),DATE(first_loan.loan_time)) BETWEEN 0 AND 30 * 14 THEN loans.loan_amt END) AS reloan_amt14
       ,SUM(CASE WHEN DATEDIFF(DATE(loans.loan_time),DATE(first_loan.loan_time)) BETWEEN 0 AND 30 * 15 THEN loans.loan_amt END) AS reloan_amt15
       ,SUM(CASE WHEN DATEDIFF(DATE(loans.loan_time),DATE(first_loan.loan_time)) BETWEEN 0 AND 30 * 16 THEN loans.loan_amt END) AS reloan_amt16
       ,SUM(CASE WHEN DATEDIFF(DATE(loans.loan_time),DATE(first_loan.loan_time)) BETWEEN 0 AND 30 * 17 THEN loans.loan_amt END) AS reloan_amt17
       ,SUM(CASE WHEN DATEDIFF(DATE(loans.loan_time),DATE(first_loan.loan_time)) BETWEEN 0 AND 30 * 18 THEN loans.loan_amt END) AS reloan_amt18
       ,SUM(CASE WHEN DATEDIFF(DATE(loans.loan_time),DATE(first_loan.loan_time)) BETWEEN 0 AND 30 * 19 THEN loans.loan_amt END) AS reloan_amt19
       ,SUM(CASE WHEN DATEDIFF(DATE(loans.loan_time),DATE(first_loan.loan_time)) BETWEEN 0 AND 30 * 20 THEN loans.loan_amt END) AS reloan_amt20
       ,SUM(CASE WHEN DATEDIFF(DATE(loans.loan_time),DATE(first_loan.loan_time)) BETWEEN 0 AND 30 * 21 THEN loans.loan_amt END) AS reloan_amt21
       ,SUM(CASE WHEN DATEDIFF(DATE(loans.loan_time),DATE(first_loan.loan_time)) BETWEEN 0 AND 30 * 22 THEN loans.loan_amt END) AS reloan_amt22
       ,SUM(CASE WHEN DATEDIFF(DATE(loans.loan_time),DATE(first_loan.loan_time)) BETWEEN 0 AND 30 * 23 THEN loans.loan_amt END) AS reloan_amt23
       ,SUM(CASE WHEN DATEDIFF(DATE(loans.loan_time),DATE(first_loan.loan_time)) BETWEEN 0 AND 30 * 24 THEN loans.loan_amt END) AS reloan_amt24

       ,SUM(CASE WHEN DATEDIFF(DATE(loans.loan_time),DATE(first_loan.loan_time)) BETWEEN 0 AND 30 * 1 AND DAY(first_loan.loan_time) < DAY(CURRENT_DATE()) THEN loans.loan_amt END) AS reloan1_amt1
       ,SUM(CASE WHEN DATEDIFF(DATE(loans.loan_time),DATE(first_loan.loan_time)) BETWEEN 0 AND 30 * 2 AND DAY(first_loan.loan_time) < DAY(CURRENT_DATE()) THEN loans.loan_amt END) AS reloan1_amt2
       ,SUM(CASE WHEN DATEDIFF(DATE(loans.loan_time),DATE(first_loan.loan_time)) BETWEEN 0 AND 30 * 3 AND DAY(first_loan.loan_time) < DAY(CURRENT_DATE()) THEN loans.loan_amt END) AS reloan1_amt3
       ,SUM(CASE WHEN DATEDIFF(DATE(loans.loan_time),DATE(first_loan.loan_time)) BETWEEN 0 AND 30 * 4 AND DAY(first_loan.loan_time) < DAY(CURRENT_DATE()) THEN loans.loan_amt END) AS reloan1_amt4
       ,SUM(CASE WHEN DATEDIFF(DATE(loans.loan_time),DATE(first_loan.loan_time)) BETWEEN 0 AND 30 * 5 AND DAY(first_loan.loan_time) < DAY(CURRENT_DATE()) THEN loans.loan_amt END) AS reloan1_amt5
       ,SUM(CASE WHEN DATEDIFF(DATE(loans.loan_time),DATE(first_loan.loan_time)) BETWEEN 0 AND 30 * 6 AND DAY(first_loan.loan_time) < DAY(CURRENT_DATE()) THEN loans.loan_amt END) AS reloan1_amt6
       ,SUM(CASE WHEN DATEDIFF(DATE(loans.loan_time),DATE(first_loan.loan_time)) BETWEEN 0 AND 30 * 7 AND DAY(first_loan.loan_time) < DAY(CURRENT_DATE()) THEN loans.loan_amt END) AS reloan1_amt7
       ,SUM(CASE WHEN DATEDIFF(DATE(loans.loan_time),DATE(first_loan.loan_time)) BETWEEN 0 AND 30 * 8 AND DAY(first_loan.loan_time) < DAY(CURRENT_DATE()) THEN loans.loan_amt END) AS reloan1_amt8
       ,SUM(CASE WHEN DATEDIFF(DATE(loans.loan_time),DATE(first_loan.loan_time)) BETWEEN 0 AND 30 * 9 AND DAY(first_loan.loan_time) < DAY(CURRENT_DATE()) THEN loans.loan_amt END) AS reloan1_amt9
       ,SUM(CASE WHEN DATEDIFF(DATE(loans.loan_time),DATE(first_loan.loan_time)) BETWEEN 0 AND 30 * 10 AND DAY(first_loan.loan_time) < DAY(CURRENT_DATE()) THEN loans.loan_amt END) AS reloan1_amt10
       ,SUM(CASE WHEN DATEDIFF(DATE(loans.loan_time),DATE(first_loan.loan_time)) BETWEEN 0 AND 30 * 11 AND DAY(first_loan.loan_time) < DAY(CURRENT_DATE()) THEN loans.loan_amt END) AS reloan1_amt11
       ,SUM(CASE WHEN DATEDIFF(DATE(loans.loan_time),DATE(first_loan.loan_time)) BETWEEN 0 AND 30 * 12 AND DAY(first_loan.loan_time) < DAY(CURRENT_DATE()) THEN loans.loan_amt END) AS reloan1_amt12
       ,SUM(CASE WHEN DATEDIFF(DATE(loans.loan_time),DATE(first_loan.loan_time)) BETWEEN 0 AND 30 * 13 AND DAY(first_loan.loan_time) < DAY(CURRENT_DATE()) THEN loans.loan_amt END) AS reloan1_amt13
       ,SUM(CASE WHEN DATEDIFF(DATE(loans.loan_time),DATE(first_loan.loan_time)) BETWEEN 0 AND 30 * 14 AND DAY(first_loan.loan_time) < DAY(CURRENT_DATE()) THEN loans.loan_amt END) AS reloan1_amt14
       ,SUM(CASE WHEN DATEDIFF(DATE(loans.loan_time),DATE(first_loan.loan_time)) BETWEEN 0 AND 30 * 15 AND DAY(first_loan.loan_time) < DAY(CURRENT_DATE()) THEN loans.loan_amt END) AS reloan1_amt15
       ,SUM(CASE WHEN DATEDIFF(DATE(loans.loan_time),DATE(first_loan.loan_time)) BETWEEN 0 AND 30 * 16 AND DAY(first_loan.loan_time) < DAY(CURRENT_DATE()) THEN loans.loan_amt END) AS reloan1_amt16
       ,SUM(CASE WHEN DATEDIFF(DATE(loans.loan_time),DATE(first_loan.loan_time)) BETWEEN 0 AND 30 * 17 AND DAY(first_loan.loan_time) < DAY(CURRENT_DATE()) THEN loans.loan_amt END) AS reloan1_amt17
       ,SUM(CASE WHEN DATEDIFF(DATE(loans.loan_time),DATE(first_loan.loan_time)) BETWEEN 0 AND 30 * 18 AND DAY(first_loan.loan_time) < DAY(CURRENT_DATE()) THEN loans.loan_amt END) AS reloan1_amt18
       ,SUM(CASE WHEN DATEDIFF(DATE(loans.loan_time),DATE(first_loan.loan_time)) BETWEEN 0 AND 30 * 19 AND DAY(first_loan.loan_time) < DAY(CURRENT_DATE()) THEN loans.loan_amt END) AS reloan1_amt19
       ,SUM(CASE WHEN DATEDIFF(DATE(loans.loan_time),DATE(first_loan.loan_time)) BETWEEN 0 AND 30 * 20 AND DAY(first_loan.loan_time) < DAY(CURRENT_DATE()) THEN loans.loan_amt END) AS reloan1_amt20
       ,SUM(CASE WHEN DATEDIFF(DATE(loans.loan_time),DATE(first_loan.loan_time)) BETWEEN 0 AND 30 * 21 AND DAY(first_loan.loan_time) < DAY(CURRENT_DATE()) THEN loans.loan_amt END) AS reloan1_amt21
       ,SUM(CASE WHEN DATEDIFF(DATE(loans.loan_time),DATE(first_loan.loan_time)) BETWEEN 0 AND 30 * 22 AND DAY(first_loan.loan_time) < DAY(CURRENT_DATE()) THEN loans.loan_amt END) AS reloan1_amt22
       ,SUM(CASE WHEN DATEDIFF(DATE(loans.loan_time),DATE(first_loan.loan_time)) BETWEEN 0 AND 30 * 23 AND DAY(first_loan.loan_time) < DAY(CURRENT_DATE()) THEN loans.loan_amt END) AS reloan1_amt23
       ,SUM(CASE WHEN DATEDIFF(DATE(loans.loan_time),DATE(first_loan.loan_time)) BETWEEN 0 AND 30 * 24 AND DAY(first_loan.loan_time) < DAY(CURRENT_DATE()) THEN loans.loan_amt END) AS reloan1_amt24

FROM
(
	SELECT  *
	FROM xyf_dws.dws_inloan_user_order_df
	WHERE pt = MAX_PT('xyf_dws.dws_inloan_user_order_df')
	AND business_line IN ('APP', '小程序端')
	AND app IN ('xyf01')
	AND loan_status = 'success'
	AND loan_flag = '首贷'
	AND DATE(loan_time) >= '2023-01-01' 
) first_loan
LEFT JOIN
(
	SELECT  *
	FROM xyf_dws.dws_inloan_user_order_df
	WHERE pt = MAX_PT('xyf_dws.dws_inloan_user_order_df')
	AND business_line IN ('APP', '小程序端')
	AND app IN ('xyf01', 'fxk')
	AND loan_status = 'success'
	AND loan_flag <> '首贷'
	AND DATE(loan_time) >= '2023-01-01' 
) loans
--cust_no level 看首贷以后的复贷 
ON first_loan.cust_no = loans.cust_no
GROUP BY  first_loan.user_no
         ,first_loan.cust_no
         ,substr(first_loan.loan_time,1,7)
         ,DATE(first_loan.loan_time)
         ,first_loan.loan_amt 
),

--=========================================================================================
--无特殊业务含义。做一个view，这样可以只改这里的driver 
--=========================================================================================
acquired_customer AS
(
SELECT  *
FROM {ltv_driver_table}  --driver  here<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<
) , 
--=========================================================================================


--账龄补齐系数,用于后续当月未走完月份的数据做放大补齐
rate as(
SELECT  a.first_loan_month
       ,b.population_category
       ,COALESCE( SUM(a.first_loan_amt) / NULLIF( SUM( CASE WHEN DAY(first_loan_date) < DAY(CURRENT_DATE()) THEN a.first_loan_amt ELSE 0 END ),0 ),0 ) AS rate1
FROM cust_level_revolving a
INNER JOIN acquired_customer b
ON a.user_no = b.app_user_id
WHERE a.first_loan_month >= '2023-06'
GROUP BY  a.first_loan_month
         ,b.population_category
)

SELECT  first_loan_month                      AS 放款月
       ,population_category
       ,first_loan_amt/10000                  AS `首贷放款(万)`
       ,SUM(reloan_amt1)/SUM(first_loan_amt)  AS MOB1
       ,SUM(reloan_amt2)/SUM(first_loan_amt)  AS MOB2
       ,SUM(reloan_amt3)/SUM(first_loan_amt)  AS MOB3
       ,SUM(reloan_amt4)/SUM(first_loan_amt)  AS MOB4
       ,SUM(reloan_amt5)/SUM(first_loan_amt)  AS MOB5
       ,SUM(reloan_amt6)/SUM(first_loan_amt)  AS MOB6
       ,SUM(reloan_amt7)/SUM(first_loan_amt)  AS MOB7
       ,SUM(reloan_amt8)/SUM(first_loan_amt)  AS MOB8
       ,SUM(reloan_amt9)/SUM(first_loan_amt)  AS MOB9
       ,SUM(reloan_amt10)/SUM(first_loan_amt) AS MOB10
       ,SUM(reloan_amt11)/SUM(first_loan_amt) AS MOB11
       ,SUM(reloan_amt12)/SUM(first_loan_amt) AS MOB12
       ,SUM(reloan_amt13)/SUM(first_loan_amt) AS MOB13
       ,SUM(reloan_amt14)/SUM(first_loan_amt) AS MOB14
       ,SUM(reloan_amt15)/SUM(first_loan_amt) AS MOB15
       ,SUM(reloan_amt16)/SUM(first_loan_amt) AS MOB16
       ,SUM(reloan_amt17)/SUM(first_loan_amt) AS MOB17
       ,SUM(reloan_amt18)/SUM(first_loan_amt) AS MOB18
       ,SUM(reloan_amt19)/SUM(first_loan_amt) AS MOB19
       ,SUM(reloan_amt20)/SUM(first_loan_amt) AS MOB20
       ,SUM(reloan_amt21)/SUM(first_loan_amt) AS MOB21
       ,SUM(reloan_amt22)/SUM(first_loan_amt) AS MOB22
       ,SUM(reloan_amt23)/SUM(first_loan_amt) AS MOB23
       ,SUM(reloan_amt24)/SUM(first_loan_amt) AS MOB24
FROM 
(
SELECT  a.first_loan_month
       ,b.population_category
       ,SUM(a.first_loan_amt)              AS first_loan_amt
       ,SUM(CASE WHEN MONTHS_BETWEEN(CONCAT(SUBSTR('{bizdate}',1,4),'-',SUBSTR('{bizdate}',5,2),'-01'),CONCAT(a.first_loan_month,'-01')) >= 2 THEN reloan_amt1 
                 WHEN MONTHS_BETWEEN(CONCAT(SUBSTR('{bizdate}',1,4),'-',SUBSTR('{bizdate}',5,2),'-01'),CONCAT(a.first_loan_month,'-01')) >= 1 THEN reloan1_amt1*rate1 END) AS reloan_amt1
       ,SUM(CASE WHEN MONTHS_BETWEEN(CONCAT(SUBSTR('{bizdate}',1,4),'-',SUBSTR('{bizdate}',5,2),'-01'),CONCAT(a.first_loan_month,'-01')) >= 3 THEN reloan_amt2 
                 WHEN MONTHS_BETWEEN(CONCAT(SUBSTR('{bizdate}',1,4),'-',SUBSTR('{bizdate}',5,2),'-01'),CONCAT(a.first_loan_month,'-01')) >= 2 THEN reloan1_amt2*rate1 END) AS reloan_amt2
       ,SUM(CASE WHEN MONTHS_BETWEEN(CONCAT(SUBSTR('{bizdate}',1,4),'-',SUBSTR('{bizdate}',5,2),'-01'),CONCAT(a.first_loan_month,'-01')) >= 4 THEN reloan_amt3 
                 WHEN MONTHS_BETWEEN(CONCAT(SUBSTR('{bizdate}',1,4),'-',SUBSTR('{bizdate}',5,2),'-01'),CONCAT(a.first_loan_month,'-01')) >= 3 THEN reloan1_amt3*rate1 END) AS reloan_amt3
       ,SUM(CASE WHEN MONTHS_BETWEEN(CONCAT(SUBSTR('{bizdate}',1,4),'-',SUBSTR('{bizdate}',5,2),'-01'),CONCAT(a.first_loan_month,'-01')) >= 5 THEN reloan_amt4 
                 WHEN MONTHS_BETWEEN(CONCAT(SUBSTR('{bizdate}',1,4),'-',SUBSTR('{bizdate}',5,2),'-01'),CONCAT(a.first_loan_month,'-01')) >= 4 THEN reloan1_amt4*rate1 END) AS reloan_amt4
       ,SUM(CASE WHEN MONTHS_BETWEEN(CONCAT(SUBSTR('{bizdate}',1,4),'-',SUBSTR('{bizdate}',5,2),'-01'),CONCAT(a.first_loan_month,'-01')) >= 6 THEN reloan_amt5 
                 WHEN MONTHS_BETWEEN(CONCAT(SUBSTR('{bizdate}',1,4),'-',SUBSTR('{bizdate}',5,2),'-01'),CONCAT(a.first_loan_month,'-01')) >= 5 THEN reloan1_amt5*rate1 END) AS reloan_amt5
       ,SUM(CASE WHEN MONTHS_BETWEEN(CONCAT(SUBSTR('{bizdate}',1,4),'-',SUBSTR('{bizdate}',5,2),'-01'),CONCAT(a.first_loan_month,'-01')) >= 7 THEN reloan_amt6 
                 WHEN MONTHS_BETWEEN(CONCAT(SUBSTR('{bizdate}',1,4),'-',SUBSTR('{bizdate}',5,2),'-01'),CONCAT(a.first_loan_month,'-01')) >= 6 THEN reloan1_amt6*rate1 END) AS reloan_amt6
       ,SUM(CASE WHEN MONTHS_BETWEEN(CONCAT(SUBSTR('{bizdate}',1,4),'-',SUBSTR('{bizdate}',5,2),'-01'),CONCAT(a.first_loan_month,'-01')) >= 8 THEN reloan_amt7 
                 WHEN MONTHS_BETWEEN(CONCAT(SUBSTR('{bizdate}',1,4),'-',SUBSTR('{bizdate}',5,2),'-01'),CONCAT(a.first_loan_month,'-01')) >= 7 THEN reloan1_amt7*rate1 END) AS reloan_amt7
       ,SUM(CASE WHEN MONTHS_BETWEEN(CONCAT(SUBSTR('{bizdate}',1,4),'-',SUBSTR('{bizdate}',5,2),'-01'),CONCAT(a.first_loan_month,'-01')) >= 9 THEN reloan_amt8 
                 WHEN MONTHS_BETWEEN(CONCAT(SUBSTR('{bizdate}',1,4),'-',SUBSTR('{bizdate}',5,2),'-01'),CONCAT(a.first_loan_month,'-01')) >= 8 THEN reloan1_amt8*rate1 END) AS reloan_amt8
       ,SUM(CASE WHEN MONTHS_BETWEEN(CONCAT(SUBSTR('{bizdate}',1,4),'-',SUBSTR('{bizdate}',5,2),'-01'),CONCAT(a.first_loan_month,'-01')) >= 10 THEN reloan_amt9 
                 WHEN MONTHS_BETWEEN(CONCAT(SUBSTR('{bizdate}',1,4),'-',SUBSTR('{bizdate}',5,2),'-01'),CONCAT(a.first_loan_month,'-01')) >= 9 THEN reloan1_amt9*rate1 END) AS reloan_amt9
       ,SUM(CASE WHEN MONTHS_BETWEEN(CONCAT(SUBSTR('{bizdate}',1,4),'-',SUBSTR('{bizdate}',5,2),'-01'),CONCAT(a.first_loan_month,'-01')) >= 11 THEN reloan_amt10 
                 WHEN MONTHS_BETWEEN(CONCAT(SUBSTR('{bizdate}',1,4),'-',SUBSTR('{bizdate}',5,2),'-01'),CONCAT(a.first_loan_month,'-01')) >= 10 THEN reloan1_amt10*rate1 END) AS reloan_amt10
       ,SUM(CASE WHEN MONTHS_BETWEEN(CONCAT(SUBSTR('{bizdate}',1,4),'-',SUBSTR('{bizdate}',5,2),'-01'),CONCAT(a.first_loan_month,'-01')) >= 12 THEN reloan_amt11 
                 WHEN MONTHS_BETWEEN(CONCAT(SUBSTR('{bizdate}',1,4),'-',SUBSTR('{bizdate}',5,2),'-01'),CONCAT(a.first_loan_month,'-01')) >= 11 THEN reloan1_amt11*rate1 END) AS reloan_amt11
       ,SUM(CASE WHEN MONTHS_BETWEEN(CONCAT(SUBSTR('{bizdate}',1,4),'-',SUBSTR('{bizdate}',5,2),'-01'),CONCAT(a.first_loan_month,'-01')) >= 13 THEN reloan_amt12 
                 WHEN MONTHS_BETWEEN(CONCAT(SUBSTR('{bizdate}',1,4),'-',SUBSTR('{bizdate}',5,2),'-01'),CONCAT(a.first_loan_month,'-01')) >= 12 THEN reloan1_amt12*rate1 END) AS reloan_amt12
       ,SUM(CASE WHEN MONTHS_BETWEEN(CONCAT(SUBSTR('{bizdate}',1,4),'-',SUBSTR('{bizdate}',5,2),'-01'),CONCAT(a.first_loan_month,'-01')) >= 14 THEN reloan_amt13 
                 WHEN MONTHS_BETWEEN(CONCAT(SUBSTR('{bizdate}',1,4),'-',SUBSTR('{bizdate}',5,2),'-01'),CONCAT(a.first_loan_month,'-01')) >= 13 THEN reloan1_amt13*rate1 END) AS reloan_amt13
       ,SUM(CASE WHEN MONTHS_BETWEEN(CONCAT(SUBSTR('{bizdate}',1,4),'-',SUBSTR('{bizdate}',5,2),'-01'),CONCAT(a.first_loan_month,'-01')) >= 15 THEN reloan_amt14 
                 WHEN MONTHS_BETWEEN(CONCAT(SUBSTR('{bizdate}',1,4),'-',SUBSTR('{bizdate}',5,2),'-01'),CONCAT(a.first_loan_month,'-01')) >= 14 THEN reloan1_amt14*rate1 END) AS reloan_amt14
       ,SUM(CASE WHEN MONTHS_BETWEEN(CONCAT(SUBSTR('{bizdate}',1,4),'-',SUBSTR('{bizdate}',5,2),'-01'),CONCAT(a.first_loan_month,'-01')) >= 16 THEN reloan_amt15 
                 WHEN MONTHS_BETWEEN(CONCAT(SUBSTR('{bizdate}',1,4),'-',SUBSTR('{bizdate}',5,2),'-01'),CONCAT(a.first_loan_month,'-01')) >= 15 THEN reloan1_amt15*rate1 END) AS reloan_amt15
       ,SUM(CASE WHEN MONTHS_BETWEEN(CONCAT(SUBSTR('{bizdate}',1,4),'-',SUBSTR('{bizdate}',5,2),'-01'),CONCAT(a.first_loan_month,'-01')) >= 17 THEN reloan_amt16 
                 WHEN MONTHS_BETWEEN(CONCAT(SUBSTR('{bizdate}',1,4),'-',SUBSTR('{bizdate}',5,2),'-01'),CONCAT(a.first_loan_month,'-01')) >= 16 THEN reloan1_amt16*rate1 END) AS reloan_amt16
       ,SUM(CASE WHEN MONTHS_BETWEEN(CONCAT(SUBSTR('{bizdate}',1,4),'-',SUBSTR('{bizdate}',5,2),'-01'),CONCAT(a.first_loan_month,'-01')) >= 18 THEN reloan_amt17 
                 WHEN MONTHS_BETWEEN(CONCAT(SUBSTR('{bizdate}',1,4),'-',SUBSTR('{bizdate}',5,2),'-01'),CONCAT(a.first_loan_month,'-01')) >= 17 THEN reloan1_amt17*rate1 END) AS reloan_amt17
       ,SUM(CASE WHEN MONTHS_BETWEEN(CONCAT(SUBSTR('{bizdate}',1,4),'-',SUBSTR('{bizdate}',5,2),'-01'),CONCAT(a.first_loan_month,'-01')) >= 19 THEN reloan_amt18 
                 WHEN MONTHS_BETWEEN(CONCAT(SUBSTR('{bizdate}',1,4),'-',SUBSTR('{bizdate}',5,2),'-01'),CONCAT(a.first_loan_month,'-01')) >= 18 THEN reloan1_amt18*rate1 END) AS reloan_amt18
       ,SUM(CASE WHEN MONTHS_BETWEEN(CONCAT(SUBSTR('{bizdate}',1,4),'-',SUBSTR('{bizdate}',5,2),'-01'),CONCAT(a.first_loan_month,'-01')) >= 20 THEN reloan_amt19 
                 WHEN MONTHS_BETWEEN(CONCAT(SUBSTR('{bizdate}',1,4),'-',SUBSTR('{bizdate}',5,2),'-01'),CONCAT(a.first_loan_month,'-01')) >= 19 THEN reloan1_amt19*rate1 END) AS reloan_amt19
       ,SUM(CASE WHEN MONTHS_BETWEEN(CONCAT(SUBSTR('{bizdate}',1,4),'-',SUBSTR('{bizdate}',5,2),'-01'),CONCAT(a.first_loan_month,'-01')) >= 21 THEN reloan_amt20 
                 WHEN MONTHS_BETWEEN(CONCAT(SUBSTR('{bizdate}',1,4),'-',SUBSTR('{bizdate}',5,2),'-01'),CONCAT(a.first_loan_month,'-01')) >= 20 THEN reloan1_amt20*rate1 END) AS reloan_amt20
       ,SUM(CASE WHEN MONTHS_BETWEEN(CONCAT(SUBSTR('{bizdate}',1,4),'-',SUBSTR('{bizdate}',5,2),'-01'),CONCAT(a.first_loan_month,'-01')) >= 22 THEN reloan_amt21 
                 WHEN MONTHS_BETWEEN(CONCAT(SUBSTR('{bizdate}',1,4),'-',SUBSTR('{bizdate}',5,2),'-01'),CONCAT(a.first_loan_month,'-01')) >= 21 THEN reloan1_amt21*rate1 END) AS reloan_amt21
       ,SUM(CASE WHEN MONTHS_BETWEEN(CONCAT(SUBSTR('{bizdate}',1,4),'-',SUBSTR('{bizdate}',5,2),'-01'),CONCAT(a.first_loan_month,'-01')) >= 23 THEN reloan_amt22 
                 WHEN MONTHS_BETWEEN(CONCAT(SUBSTR('{bizdate}',1,4),'-',SUBSTR('{bizdate}',5,2),'-01'),CONCAT(a.first_loan_month,'-01')) >= 22 THEN reloan1_amt22*rate1 END) AS reloan_amt22
       ,SUM(CASE WHEN MONTHS_BETWEEN(CONCAT(SUBSTR('{bizdate}',1,4),'-',SUBSTR('{bizdate}',5,2),'-01'),CONCAT(a.first_loan_month,'-01')) >= 24 THEN reloan_amt23 
                 WHEN MONTHS_BETWEEN(CONCAT(SUBSTR('{bizdate}',1,4),'-',SUBSTR('{bizdate}',5,2),'-01'),CONCAT(a.first_loan_month,'-01')) >= 23 THEN reloan1_amt23*rate1 END) AS reloan_amt23
       ,SUM(CASE WHEN MONTHS_BETWEEN(CONCAT(SUBSTR('{bizdate}',1,4),'-',SUBSTR('{bizdate}',5,2),'-01'),CONCAT(a.first_loan_month,'-01')) >= 25 THEN reloan_amt24 
                 WHEN MONTHS_BETWEEN(CONCAT(SUBSTR('{bizdate}',1,4),'-',SUBSTR('{bizdate}',5,2),'-01'),CONCAT(a.first_loan_month,'-01')) >= 24 THEN reloan1_amt24*rate1 END) AS reloan_amt24

FROM cust_level_revolving a
INNER JOIN acquired_customer b
ON a.user_no = b.app_user_id
INNER JOIN rate c
ON a.first_loan_month = c.first_loan_month AND b.population_category = c.population_category
WHERE a.first_loan_month >= '2024-01'
GROUP BY  a.first_loan_month
         ,b.population_category
)
GROUP BY  first_loan_month
         ,population_category
         ,first_loan_amt
'''.format(ltv_driver_table=ltv_driver_table, bizdate = bizdate)

data_stats = qat.run_query(query)

正在获取数据，首段 SQL: 
--=============================================== ...


In [5]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
from IPython.display import display

def normalize_month_str(x):
    """把 '2024/01' / '2024-01-15' 等写法统一成 'YYYY-MM'."""
    dt = pd.to_datetime(str(x).replace("/", "-"), errors="coerce")
    return dt.strftime("%Y-%m") if pd.notna(dt) else str(x)


def is_mob_col(col_name):
    """判断列名是否形如 mob1 / MOB12."""
    s = str(col_name).strip().lower()
    return s.startswith("mob") and s[3:].isdigit()


def mob_num(col_name):
    """从 'mob5' / 'MOB18' 里取出 5 / 18."""
    return int(str(col_name).strip().lower().replace("mob", ""))


def get_reloan_mob_cols(df: pd.DataFrame) -> list[str]:
    """按 MOB 编号升序返回 mob 列名。"""
    return sorted([c for c in df.columns if is_mob_col(c)], key=mob_num)


def resolve_diagonal_month(
    df_wide: pd.DataFrame,
    mob_cols: list[str],
    diagonal_month: str | None = None,
    *,
    month_col: str = "放款月",
):
    """把对角线锚点统一转换成 Period。

    未传月份时，默认取 mob1 最后一个非空月，兼容历史调用方式。
    """
    loan_periods = pd.to_datetime(df_wide[month_col], errors="coerce").dt.to_period("M")
    if diagonal_month is None:
        mob1_col = mob_cols[0] if mob_cols else None
        return (
            loan_periods[df_wide[mob1_col].notna()].max()
            if mob1_col is not None and df_wide[mob1_col].notna().any()
            else pd.NaT
        )
    return pd.Period(normalize_month_str(diagonal_month), freq="M")


def is_on_or_before_diagonal(loan_period, mob_k: int, diagonal_period) -> bool:
    """判断某个点是否位于指定对角线内。"""
    return bool(pd.notna(loan_period) and pd.notna(diagonal_period) and (loan_period + (mob_k - 1)) <= diagonal_period)


def fill_reloan_triangle(
    df_wide: pd.DataFrame,
    fill_start_month: str | None = None,
    *,
    actual_mob1_completed_month: str | None = None,
    month_col: str = "放款月",
):
    """按“指定填充对角线 + 原递推逻辑”补全复贷三角。

    - fill_start_month：填充起始月。以这条对角线为界，对角线内保留原始值，
      对角线外按原逻辑外推。
    - actual_mob1_completed_month：绘图时划分实际值/预测值的对角线。
      若不传，默认沿用 fill_start_month。
    """
    # 1) 标准化并排序，保证递推顺序稳定
    df = df_wide.copy().reset_index(drop=True)
    df[month_col] = df[month_col].map(normalize_month_str)
    df = df.sort_values(month_col).reset_index(drop=True)

    # 2) 识别 mob 列，并统一转数值
    mob_cols = get_reloan_mob_cols(df)
    for col in mob_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    if not mob_cols:
        empty_mask = pd.DataFrame(False, index=df.index, columns=[])
        return df, empty_mask, empty_mask

    # 若只传了 actual_mob1_completed_month，则填充起始月默认与其一致。
    effective_fill_start_month = fill_start_month if fill_start_month is not None else actual_mob1_completed_month
    effective_actual_month = actual_mob1_completed_month if actual_mob1_completed_month is not None else effective_fill_start_month

    fill_period = resolve_diagonal_month(
        df,
        mob_cols,
        diagonal_month=effective_fill_start_month,
        month_col=month_col,
    )
    actual_period = resolve_diagonal_month(
        df,
        mob_cols,
        diagonal_month=effective_actual_month,
        month_col=month_col,
    )
    loan_periods = pd.to_datetime(df[month_col], errors="coerce").dt.to_period("M")

    # 3) 先按 fill_start_month 划对角线：只保留对角线内的原始值
    filled = df.copy()
    for row_idx, loan_period in enumerate(loan_periods):
        for col in mob_cols:
            if not is_on_or_before_diagonal(loan_period, mob_num(col), fill_period):
                filled.at[row_idx, col] = np.nan

    # 4) MOB1：空值 = 前两个月算术平均
    mob1_col = mob_cols[0]
    for row_idx in range(2, len(filled)):
        if pd.isna(filled.at[row_idx, mob1_col]):
            prev_1 = filled.at[row_idx - 1, mob1_col]
            prev_2 = filled.at[row_idx - 2, mob1_col]
            if pd.notna(prev_1) and pd.notna(prev_2):
                filled.at[row_idx, mob1_col] = (prev_1 + prev_2) / 2

    # 5) MOB2+：沿用原来的三角递推逻辑
    for mob_idx in range(1, len(mob_cols)):
        cur_col = mob_cols[mob_idx]
        prev_col = mob_cols[mob_idx - 1]

        for row_idx in range(2, len(filled)):
            if pd.notna(filled.at[row_idx, cur_col]):
                continue

            cur_same_month_prev_mob = filled.at[row_idx, prev_col]
            cur_prev_1 = filled.at[row_idx - 1, cur_col]
            cur_prev_2 = filled.at[row_idx - 2, cur_col]
            prev_prev_1 = filled.at[row_idx - 1, prev_col]
            prev_prev_2 = filled.at[row_idx - 2, prev_col]

            needed = [
                cur_same_month_prev_mob,
                cur_prev_1,
                cur_prev_2,
                prev_prev_1,
                prev_prev_2,
            ]
            if all(pd.notna(v) for v in needed):
                avg_increment = ((cur_prev_2 + cur_prev_1) - (prev_prev_2 + prev_prev_1)) / 2
                filled.at[row_idx, cur_col] = cur_same_month_prev_mob + avg_increment

    # 6) 绘图时，再单独根据 actual_mob1_completed_month 判断实际值/预测值
    actual_mask = pd.DataFrame(False, index=filled.index, columns=mob_cols)
    predict_mask = pd.DataFrame(False, index=filled.index, columns=mob_cols)
    for row_idx, loan_period in enumerate(loan_periods):
        for col in mob_cols:
            is_actual = is_on_or_before_diagonal(loan_period, mob_num(col), actual_period)
            has_value = pd.notna(filled.at[row_idx, col])
            actual_mask.at[row_idx, col] = is_actual and has_value
            predict_mask.at[row_idx, col] = (not is_actual) and has_value

    return filled, actual_mask, predict_mask


# =================================================================
# 复贷曲线绘图
# =================================================================
def plot_reloan_curve_single_population(
    df_pop: pd.DataFrame,
    title: str,
    *,
    month_col: str = "放款月",
    mob_min: int = 1,
    mob_max: int = 18,
    recent_months: int | None = None,
    selected_months: list[str] | None = None,
    start_month: str | None = None,
    end_month: str | None = None,
    fill_start_month: str | None = None,
    actual_mob1_completed_month: str | None = None,
    plot_mode: str = "actual_and_filled",
    colors: dict = None,
    actual_linestyle: str = "-",
    fill_linestyle: str = "--",
    marker: str = "o",
    linewidth: float = 1.5,
    markersize: float = 3,
    figsize: tuple = (12, 8),
    dpi: int = 300,
    ylim: tuple = None,
    y_label: str = "复贷系数",
    legend_title: str = "放款月",
    legend_loc: str = "upper left",
    legend_bbox_to_anchor: tuple = (1.02, 1),
    grid_axis: str = "y",
    grid_linestyle: str = "--",
    grid_alpha: float = 0.3,
    save_dir: str = None,
    suffix: str = ".png",
    bbox_inches: str = "tight",
    show_inline: bool = True,
):
    if df_pop is None or len(df_pop) == 0:
        raise ValueError("df_pop 为空，无法绘图")
    if plot_mode not in {"actual_only", "actual_and_filled"}:
        raise ValueError("plot_mode 只能是 'actual_only' 或 'actual_and_filled'")
    if mob_min < 1 or mob_max < mob_min:
        raise ValueError("mob_min/mob_max 设置不合法")

    if save_dir is None:
        save_dir = os.path.join(os.getcwd(), "imgs")
    os.makedirs(save_dir, exist_ok=True)

    plt.rcParams["font.sans-serif"] = ["STKaiti", "Kaiti"]
    plt.rcParams["axes.unicode_minus"] = False

    default_palette = [
        "#2F6EEB", "#F39C12", "#9e9e9e", "#F1C40F", "#34B4F4",
        "#6A5ACD", "#1ABC9C", "#D35400", "#E74C3C", "#000000",
    ]

    # 填充和绘图统一走 fill_reloan_triangle，
    # 但 fill_start_month 与 actual_mob1_completed_month 可以分开传入。
    filled_wide, actual_mask, predict_mask = fill_reloan_triangle(
        df_pop,
        fill_start_month=fill_start_month,
        actual_mob1_completed_month=actual_mob1_completed_month,
        month_col=month_col,
    )

    mob_cols = [c for c in get_reloan_mob_cols(filled_wide) if mob_min <= mob_num(c) <= mob_max]
    if not mob_cols:
        raise ValueError(f"找不到指定范围内的 MOB 列，可用列为 {filled_wide.columns.tolist()}")

    filled_wide = filled_wide.reset_index(drop=True)
    actual_mask = actual_mask.loc[:, mob_cols].reset_index(drop=True)
    predict_mask = predict_mask.loc[:, mob_cols].reset_index(drop=True)

    month_period = pd.to_datetime(filled_wide[month_col], errors="coerce").dt.to_period("M")
    row_mask = month_period.notna()
    if selected_months:
        selected_periods = {pd.Period(normalize_month_str(m), freq="M") for m in selected_months}
        row_mask &= month_period.isin(selected_periods)
    else:
        if start_month:
            row_mask &= month_period >= pd.Period(start_month, freq="M")
        if end_month:
            row_mask &= month_period <= pd.Period(end_month, freq="M")
        if recent_months is not None:
            anchor = month_period[row_mask].max()
            if pd.notna(anchor):
                row_mask &= month_period >= anchor - (recent_months - 1)

    filled_wide = filled_wide.loc[row_mask].reset_index(drop=True)
    actual_mask = actual_mask.loc[row_mask].reset_index(drop=True)
    predict_mask = predict_mask.loc[row_mask].reset_index(drop=True)

    if filled_wide.empty:
        raise ValueError("筛选后无数据可绘制")

    fig, ax = plt.subplots(figsize=figsize, dpi=dpi)
    x = np.arange(1, len(mob_cols) + 1)
    mob_labels = [f"MOB{mob_num(c)}" for c in mob_cols]
    has_label = False

    for idx, (_, row) in enumerate(filled_wide.iterrows()):
        month = row[month_col]
        color = colors.get(month) if colors else default_palette[idx % len(default_palette)]

        y = row[mob_cols].to_numpy(dtype=float)
        row_actual_mask = actual_mask.iloc[idx][mob_cols].to_numpy(dtype=bool) & ~np.isnan(y)
        row_predict_mask = predict_mask.iloc[idx][mob_cols].to_numpy(dtype=bool) & ~np.isnan(y)

        if row_actual_mask.any():
            ax.plot(
                x[row_actual_mask], y[row_actual_mask],
                label=month, color=color,
                linestyle=actual_linestyle, marker=marker,
                linewidth=linewidth, markersize=markersize,
            )
            has_label = True

        if plot_mode == "actual_and_filled" and row_predict_mask.any():
            predict_idx = np.where(row_predict_mask)[0]
            x_dash = x[predict_idx].tolist()
            y_dash = y[predict_idx].tolist()

            actual_idx = np.where(row_actual_mask)[0]
            if actual_idx.size > 0:
                last_actual = actual_idx.max()
                if last_actual < predict_idx.min():
                    x_dash = [x[last_actual]] + x_dash
                    y_dash = [y[last_actual]] + y_dash

            ax.plot(
                x_dash, y_dash, color=color,
                linestyle=fill_linestyle, marker=marker,
                linewidth=linewidth, markersize=markersize,
            )

    ax.set_title(title, fontsize=15, fontweight="bold")
    ax.set_xlabel("MOB")
    ax.set_ylabel(y_label)
    ax.set_xticks(x)
    ax.set_xticklabels(mob_labels)
    ax.yaxis.set_major_formatter(PercentFormatter(1.0))
    ax.grid(True, axis=grid_axis, linestyle=grid_linestyle, alpha=grid_alpha)
    if ylim is not None:
        ax.set_ylim(*ylim)
    if has_label:
        ax.legend(title=legend_title, bbox_to_anchor=legend_bbox_to_anchor, loc=legend_loc)

    fig.tight_layout()

    safe_name = re.sub(r'[<>:"/\\|?*]+', "_", title).strip()
    img_path = os.path.join(save_dir, f"{safe_name}{suffix}")
    fig.savefig(img_path, dpi=dpi, bbox_inches=bbox_inches)
    if show_inline:
        display(fig)
    plt.close(fig)
    return img_path


In [13]:
# 两个参数相同则填充与绘图口径一致；不同则可分开控制。
fill_start_month = "2026-04"
actual_mob1_completed_month = "2026-03"

for pop, sheet in population_to_sheet.items():
    sub = data_stats.loc[data_stats["population_category"] == pop].copy()
    if sub.empty:
        print(f"[跳过] {pop}: 无数据")
        continue

    sub["放款月"] = sub["放款月"].astype(str)
    sub = sub.sort_values("放款月").reset_index(drop=True)

    # 先 copy 真实值，再向对角线外扩充
    sub, _actual_mask, _predict_mask = fill_reloan_triangle(
        sub,
        fill_start_month=fill_start_month,
        actual_mob1_completed_month=actual_mob1_completed_month,
        month_col="放款月",
    )

    df_t_pop = sub.T.reset_index()

    qat.write_dataframe_to_template_area(
        file_path=file_path,
        sheet_name=sheet,
        df=df_t_pop,
        start_cell="B26",
        include_header=False,
        clear_before_write=False,
    )

    print(f"[完成] {pop} -> {sheet}")


# for pop in population_to_sheet.keys():
#     df_pop = data_stats.loc[data_stats["population_category"] == pop].copy()
#     if df_pop.empty:
#         print(f"[跳过] {pop}: 无数据")
#         continue

#     img_path = plot_reloan_curve_single_population(
#         df_pop=df_pop,
#         title=f"{pop}复贷系数",
#         start_month="2025-08",
#         end_month="2026-04",
#         fill_start_month="2026-04",
#         actual_mob1_completed_month="2026-03",
#         plot_mode="actual_and_filled",
#         save_dir=r"D:\10.LTV月度更新\imgs",
#         ylim=(0, 2),
#     )
#     print(img_path)

[OK] 写入 27 行 x 31 列 -> sheet=APP整体 B26
[完成] APP整体 -> APP整体
[OK] 写入 27 行 x 31 列 -> sheet=抖音 B26
[完成] 信息流-DY -> 抖音
[OK] 写入 27 行 x 31 列 -> sheet=广点通 B26
[完成] 信息流-TX -> 广点通
[OK] 写入 27 行 x 31 列 -> sheet=短信 B26
[完成] 短信 -> 短信
[OK] 写入 27 行 x 5 列 -> sheet=尊享贷 B26
[完成] 尊享贷 -> 尊享贷


In [21]:
# 常见调用案例

# 假设前面已经有：
# df_app = data_stats.loc[data_stats["population_category"] == "APP整体"].copy()

# # 1. 只画最近12个月的真实值
# img_path = plot_reloan_curve_single_population(
#     df_pop=df_app,
#     title="APP整体复贷系数",
#     recent_months=18,
#     actual_mob1_completed_month="2026-02",
#     mob_min=1,
#     mob_max=18,
#     plot_mode="actual_only",
#     ylim=(0, 2.2),
# )
# print(img_path)

# 2. 画最近6个月，真实值+填充值
# img_path = plot_reloan_curve_single_population(
#     df_pop=df_app,
#     title="APP整体复贷系数",
#     start_month="2025-07",
#     end_month="2026-02",
#     actual_mob1_completed_month="2026-02",
#     plot_mode="actual_and_filled",
#     ylim=(0, 2.2),
# )
# print(img_path)

# 3. 只看指定几个月
# img_path = plot_reloan_curve_single_population(
#     df_pop=df_app,
#     title="APP整体复贷系数_指定月份",
#     selected_months=["2025-01", "2025-06", "2026-02"],
#     mob_min=1,
#     mob_max=18,
#     actual_mob1_completed_month="2026-02",
#     plot_mode="actual_and_filled",
#     ylim=(0, 2.2),
# )
# print(img_path)

# 4. 只看指定区间
# img_path = plot_reloan_curve_single_population(
#     df_pop=df_app,
#     title="APP整体复贷系数_区间查看",
#     recent_months=None,
#     start_month="2025-07",
#     end_month="2026-02",
#     mob_min=1,
#     mob_max=18,
#     actual_mob1_completed_month="2026-02",
#     plot_mode="actual_and_filled",
#     ylim=(0, 2.2),
# )
# print(img_path)

# 5. 只看前12个MOB
# img_path = plot_reloan_curve_single_population(
#     df_pop=df_app,
#     title="APP整体复贷系数_MOB1到MOB12",
#     recent_months=6,
#     mob_min=1,
#     mob_max=12,
#     actual_mob1_completed_month="2026-02",
#     plot_mode="actual_and_filled",
#     ylim=(0, 1.5),
# )
# print(img_path)

# 6. 自定义颜色和线型
# img_path = plot_reloan_curve_single_population(
#     df_pop=df_app,
#     title="APP整体复贷系数_自定义样式",
#     selected_months=["2025-11", "2025-12", "2026-01", "2026-02"],
#     mob_min=1,
#     mob_max=18,
#     actual_mob1_completed_month="2026-02",
#     plot_mode="actual_and_filled",
#     colors={
#         "2025-11": "#2F6EEB",
#         "2025-12": "#F39C12",
#         "2026-01": "#16A085",
#         "2026-02": "#E74C3C",
#     },
#     actual_linestyle="-",
#     fill_linestyle="--",
#     marker="o",
#     linewidth=2.5,
#     markersize=5,
#     figsize=(14, 8),
#     dpi=200,
#     ylim=(0, 2.2),
# )
# print(img_path)

# 7. 指定保存目录，并在 notebook 显示
# img_path = plot_reloan_curve_single_population(
#     df_pop=df_app,
#     title="APP整体复贷系数_保存到指定目录",
#     recent_months=6,
#     mob_min=1,
#     mob_max=18,
#     actual_mob1_completed_month="2026-02",
#     plot_mode="actual_and_filled",
#     save_dir=r"D:\10.LTV月度更新\imgs",
#     show_inline=True,
#     ylim=(0, 2.2),
# )
# print(img_path)

# 8. 外部循环批量出图
# for pop in population_to_sheet.keys():
#     df_pop = data_stats.loc[data_stats["population_category"] == pop].copy()
#     if df_pop.empty:
#         continue
#
#     img_path = plot_reloan_curve_single_population(
#         df_pop=df_pop,
#         title=f"{pop}复贷系数",
#         recent_months=6,
#         mob_min=1,
#         mob_max=18,
#         actual_mob1_completed_month="2026-02",
#         plot_mode="actual_and_filled",
#         save_dir=r"D:\10.LTV月度更新\imgs",
#         ylim=(0, 2.2),
#     )
#     print(img_path)

#### 计算久期余额
##### 首贷久期

In [ ]:
WITH driver AS
(
    SELECT DISTINCT
           population_category
          ,TRIM(split_order_number) AS order_number
    FROM
    (
        SELECT  order_number
               ,population_category
        FROM ltv_first_loan_driver_lss

        UNION ALL

        SELECT  order_number
               ,'APP复贷大盘' AS population_category
        FROM xyf_dws.dws_inloan_user_order_df
        WHERE pt = MAX_PT('xyf_dws.dws_inloan_user_order_df')
        AND loan_status = 'success'
        AND app IN ('xyf01', 'fxk')
        AND loan_flag <> '首贷'
        AND business_line IN ('APP', '小程序端')
        AND DATE(loan_time) >= '2024-01-01'
    ) t
    LATERAL VIEW OUTER EXPLODE(SPLIT(order_number, ',')) o AS split_order_number
)

SELECT  d.population_category
       ,a.pt                                                                  AS 统计日期
       ,TO_CHAR(a.loan_time,'yyyy-MM')                                        AS 放款日期
       ,a.period                                                              AS 期数
       ,FLOOR(months_between(to_date(a.pt,'yyyyMMdd'),to_date(a.loan_time)))  AS mob
       ,b.day_id_iso                                                          AS data_date2
       ,SUM(a.loan_amt)                                                       AS 放款金额
       ,SUM(CASE WHEN a.overdue_days <= 0 THEN a.loan_balance END)            AS M0金额
       ,SUM(CASE WHEN a.overdue_days <= 30 THEN a.loan_balance END)           AS M1minus金额
       ,SUM(CASE WHEN a.overdue_days <= 90 THEN a.loan_balance END)           AS M3minus金额
       ,SUM(CASE WHEN a.overdue_days <= 180 THEN a.loan_balance END)          AS M6minus金额
FROM
(
    SELECT *
    FROM xyf_dws.dws_repay_user_order_df
    WHERE pt IN (
        '20240131', '20240229', '20240331', '20240430', '20240531', '20240630',
        '20240731', '20240831', '20240930', '20241031', '20241130', '20241231',
        '20250131', '20250228', '20250331', '20250430', '20250531', '20250630',
        '20250731', '20250831', '20250930', '20251031', '20251130', '20251231',
        '20260131', '20260228', '20260331', '20260430', '20260531'
    )
    AND FLOOR(months_between(to_date(pt, 'yyyyMMdd'), to_date(loan_time))) BETWEEN 0 AND 19
) a
INNER JOIN xyf_dim.dim_pub_date b
ON a.pt = b.day_id  
AND b.is_lastday = 1   --月末 
AND b.day_id_iso >= '2023-01-31'
AND b.day_id_iso <= DATE_SUB(DATETRUNC(CURRENT_DATE(), 'MM'), 1)  --上月末最后一天 

INNER JOIN driver d
ON a.order_number = d.order_number

GROUP BY  d.population_category
         ,a.pt
         ,TO_CHAR(a.loan_time,'yyyy-MM')
         ,a.period
         ,FLOOR(months_between(to_date(a.pt,'yyyyMMdd'),to_date(a.loan_time)))
         ,b.day_id_iso
;